# Semana 07: Intensivao de Node-RED na Pratica IIoT

## Construcao Passo a Passo de um Pipeline Completo de Ingestao, Roteamento e Tratamento de Dados Industriais

Este caderno didatico constitui o guia de bancada para a **Semana 07** da disciplina de **Automacao Industrial**. O objetivo desta sessao e construir do zero absoluto o pipeline industrial de dados formalizado no arquivo `materiais/semana_05/nodered/flows_semana05.json`.

### Contexto Operacional da Planta
Nesta etapa do curso, a camada de aquisicao do chao de fabrica (OT) ja possui um publicador continuo em operacao na rede industrial, injetando pacotes telemetricos via protocolo MQTT no broker da planta. Nosso foco e atuar na camada intermediaria (Edge/Middleware) utilizando o **Node-RED** para:
1. Estabelecer a subscricao multinivel no broker;
2. Desserializar payloads textuais em objetos estruturados;
3. Rotear fluxos de acordo com a semantica dos topicos industriais;
4. Executar algoritmos de classificacao de criticidade e calculo de indicadores de manufatura;
5. Estruturar a observabilidade atraves de multiplos nos de depuracao seletivos.

---

## 1. Arquitetura do Pipeline e Topologia de Fluxo

O pipeline implementado em `flows_semana05.json` segue o padrao arquitetural de passagem de mensagens baseada em fluxos (Flow-Based Programming - FBP). Cada componente atua como uma unidade funcional estritamente delimitada, manipulando o envelope de mensagem universal denominado `msg`.

```
Topologia Logica do Fluxo a ser Construido:

  [ Publicador MQTT ]               [ Injetor Manual ]
           |                                 |
           v (fabrica/#)                     v (teste manual)
    [ mqtt in ] ------------------------> [ switch ]
           |                                 |
           +--> [ debug (Bruto) ]            |-- (telemetria) -> [ function: Telemetria ] -> [ debug ]
           |                                 |-- (alarmes)    -> [ function: Alarme ]     -> [ debug ]
           v                                 |-- (producao)   -> [ function: Producao ]   -> [ debug ]
       [ json ] ----------------------------+-- (residual)   -> [ debug (Outros) ]
```

### Elementos Centrais do Ciclo de Vida da Mensagem
- `msg.topic`: Chave textual hierarquica contendo a rota de publicacao no padrao MQTT (ex: `fabrica/linha1/prensa01/telemetria`).
- `msg.payload`: Carga util da mensagem. Ao chegar pelo no `mqtt in`, o payload e transmitido como uma cadeia de caracteres (string serializada). Apos processamento no no `json`, ele passa a ser um objeto JavaScript manipulavel por chave e valor (ex: `msg.payload.temperatura`).

---

## 2. Verificacao de Infraestrutura e Conectividade de Rede

Antes de iniciar o desenvolvimento na interface visual do Node-RED, e necessario validar se as portas dos servicos essenciais estao abertas e respondendo no ambiente operacional:
- **Porta 1883:** Broker MQTT (Eclipse Mosquitto);
- **Porta 1880:** Servidor de Aplicacao do Node-RED.

Execute a celula Python a seguir para confirmar a integridade dos sockets TCP locais.

In [ ]:
# Validacao de Conectividade de Rede via Sockets TCP
import socket

def verificar_servico(host, porta, nome_servico):
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(2.0)
    try:
        sock.connect((host, porta))
        print(f"[STATUS OPERACIONAL] {nome_servico} ativo e acessivel em {host}:{porta}")
        sock.close()
        return True
    except Exception as erro:
        print(f"[FALHA DE CONEXAO] {nome_servico} inacessivel em {host}:{porta} -> Motivo: {erro}")
        return False

print("--- DIAGNOSTICO DE PORTAS DE REDE ---")
status_broker = verificar_servico("localhost", 1883, "Broker MQTT (Mosquitto)")
status_nodered = verificar_servico("localhost", 1880, "Servidor Node-RED")

if status_broker and status_nodered:
    print("\nInfraestrutura pronta. Prossiga para a interface web do Node-RED em http://localhost:1880")
else:
    print("\nAtencao: Verifique se os servicos estao iniciados (ex: via Docker Compose na pasta materiais/semana_05).")


--- 

## 3. Passo 1: Inicializacao da Workspace e No de Configuracao do Broker

O Node-RED organiza seus projetos em abas (*workspaces*). Cada aba contem um grafo independente de execucao, compartilhando recursos globais denominados **Configuration Nodes**.

### 3.1 Criando uma Nova Aba de Trabalho
1. Abra o navegador web no endereco: **http://localhost:1880**
2. Localize a barra superior de abas e clique no botao `+` (adicionar aba).
3. De um duplo clique sobre o nome da aba recem-criada para abrir suas propriedades.
4. Defina o campo **Name / Label** como: `Semana 07 - Intensivao Node-RED`.
5. Clique em **Done**.

### 3.2 Criando o No Global de Configuracao do Broker (mqtt-broker)
O no de configuracao do broker centraliza a sessao TCP com o servidor MQTT. Todos os nos de entrada e saida reutilizam esta mesma conexao logica.

1. Na paleta a esquerda, no grupo **Network**, arraste um no **mqtt in** para a area de trabalho.
2. De um duplo clique sobre o no para abrir o painel de edicao.
3. No campo **Server**, selecione a opcao **Add new mqtt-broker...** e clique no icone de lapis para editar.
4. Configure os parametros da aba **Connection**:
   - **Name:** `Mosquitto Local`
   - **Server:** `mosquitto` (caso o Node-RED esteja rodando dentro do Docker) ou `localhost` (caso esteja executando nativamente na maquina hospedeira).
   - **Port:** `1883`
   - **Client ID:** `nodered_subscriber_lab`
   - **Protocol Version:** `MQTT V3.1.1` (versao padrao para compatibilidade industrial)
   - **Auto-connect:** Habilitado
   - **Keep alive:** `60` segundos
   - **Clean session:** Habilitado (garante que inscricoes anteriores nao gerem fila residual em reinicializacoes de teste)

5. Configure a aba **Messages** (Mensagens de Ciclo de Vida do Cliente):
   O protocolo MQTT permite monitorar a presenca dos nos na rede atraves de mensagens automaticas de conexao e desconexao anomala (LWT - *Last Will and Testament*):
   - **Birth Message (Mensagem de Inicializacao):**
     - Topic: `nodered/status`
     - QoS: `0`
     - Payload: `{"status":"ONLINE","origem":"Node-RED"}`
   - **Close Message (Mensagem de Encerramento Gracioso):**
     - Topic: `nodered/status`
     - QoS: `0`
     - Payload: `{"status":"OFFLINE","origem":"Node-RED"}`
   - **Will Message (Testamento por Perda Abrupta de Conexao - LWT):**
     - Topic: `nodered/status`
     - QoS: `0`
     - Payload: `{"status":"CRASHED","origem":"Node-RED"}`

6. Clique em **Add** para salvar a configuracao do broker.

--- 

## 4. Passo 2: Parametrizacao do No de Entrada (mqtt in)

Com o no de conexao global definido, agora finalizamos a configuracao do no de subscricao na area de transferencia.

### 4.1 Configuracao dos Campos do No
- **Server:** `Mosquitto Local` (selecione a configuracao criada no Passo 1)
- **Action:** `Subscribe to single topic`
- **Topic:** `fabrica/#`
- **QoS:** `0` (Qualidade de servico com entrega *at most once*, recomendada para telemetria de alta frequencia)
- **Output:** `auto-detect` (ou `a String`)
- **Name:** `MQTT In: fabrica/#`

### 4.2 Fundamentacao do Caractere Curinga Multinivel (#)
No padrao MQTT, o caractere sustenido (`#`) e um curinga multinivel (*multi-level wildcard*) que deve ser posicionado obrigatoriamente no final da arvore de topicos. Ele instrui o broker a encaminhar qualquer mensagem cujo topico inicie com `fabrica/`, independentemente da profundidade hierarquica:
- `fabrica/linha1/prensa01/telemetria` (capturado)
- `fabrica/linha1/prensa01/alarmes` (capturado)
- `fabrica/linha2/compressor/producao` (capturado)
- `predio/energia` (ignorado, fora do escopo)

Clique em **Done** para concluir a parametrizacao do no de entrada.

--- 

## 5. Passo 3: No de Desserializacao Estruturada (JSON Parser)

Quando uma mensagem MQTT chega ao Node-RED, sua carga util (`msg.payload`) e recebida como uma sequencia de bytes codificados em UTF-8 (texto puro). Nessa forma, os atributos internos nao podem ser acessados diretamente por notacao de ponto (ex: `msg.payload.temperatura` resultaria em erro ou valor indefinido).

O no **json** realiza a conversao bidirecional entre texto JSON e objeto JavaScript em memoria heap.

### 5.1 Insercao e Parametrizacao
1. Na paleta a esquerda, no grupo **Parser**, localize e arraste o no **json** para o canvas.
2. Posicione-o a direita do no `MQTT In: fabrica/#`.
3. De um duplo clique sobre o no para abrir suas propriedades:
   - **Action:** `Always convert to JavaScript Object`
   - **Property:** `payload`
   - **Name:** `JSON Parser`
4. Clique em **Done**.

### 5.2 Conexao Fisica (Wire)
- Conecte a saida do no `MQTT In: fabrica/#` a entrada do no `JSON Parser`.

--- 

## 6. Passo 4: Ponto de Observabilidade de Entrada (Debug Bruto)

Em arquiteturas de integracao de dados fabris, a primeira boa pratica de observabilidade consiste em criar um ponto de derivacao (*tap*) antes de qualquer mutacao logica. Isso permite comprovar se eventuais anomalias provem da fonte de dados ou do processamento interno.

### 6.1 Insercao do No de Depuracao Bruta
1. No grupo **Common**, arraste um no **debug** para o canvas.
2. Posicione-o abaixo do no `JSON Parser`.
3. De um duplo clique sobre ele e preencha:
   - **Output:** `msg.payload`
   - **To:** `debug window` (painel lateral de depuracao)
   - **Name:** `[DEBUG] Todas Msg Brutas`
4. Clique em **Done**.

### 6.2 Conexao (Wire de Derivacao)
- Conecte a saida do no `MQTT In: fabrica/#` (ou a saida do `JSON Parser`) na entrada do no `[DEBUG] Todas Msg Brutas`.

--- 

## 7. Passo 5: Multiplexador e Roteador de Fluxos (Switch)

O no **switch** opera como um demultiplexador condicional. Ele avalia propriedades da mensagem recebida em relacao a uma lista de predicados e direciona o fluxo para portas de saida especificas.

No nosso fluxo, o roteamento sera baseado no conteudo da string `msg.topic`.

### 7.1 Insercao e Configuracao Geral
1. No grupo **Function**, arraste um no **switch** para o canvas.
2. Posicione-o a direita do no `JSON Parser`.
3. De um duplo clique para abrir as propriedades:
   - **Property:** `msg.topic`
   - **Name:** `Roteador por Topico/Tipo`

### 7.2 Definicao da Tabela de Regras e Portas de Saida
Clique no botao `+ add` no rodape da janela para configurar 4 regras ordenadas:

| Porta | Operador Condicional | Argumento | Tipo | Descricao Tecnica |
| :--- | :--- | :--- | :--- | :--- |
| **Porta 1** | `contains` | `telemetria` | `String` | Filtra variaveis continuas (temperatura, vibracao, pressao) |
| **Porta 2** | `contains` | `alarmes` | `String` | Filtra eventos discretos de seguranca operacional |
| **Porta 3** | `contains` | `producao` | `String` | Filtra contadores de pecas e status de linha |
| **Porta 4** | `otherwise` | *(sem argumento)* | Residual | Regra de descarte ou tratamento residual (fallback) |

Certifique-se de que a opcao inferior esteja marcada como: **checking all rules** (avaliacao de todas as regras).

4. Clique em **Done**.

### 7.3 Conexao (Wire)
- Conecte a saida do no `JSON Parser` a entrada do no `Roteador por Topico/Tipo`.

--- 

## 8. Passo 6: No de Funcao 1 - Tratamento e Limiares de Telemetria

O no **function** permite a execucao de blocos de script em JavaScript puro dentro da sandbox do motor V8 do Node.js. O objeto `msg` e recebido como parametro de entrada e deve ser devolvido atraves da instrucao `return msg;`.

Este primeiro no de funcao conectara na **Porta 1** do no `switch`.

### 8.1 Objetivo do Algoritmo
1. Extrair os valores numericos com fallback para zero caso venham nulos ou indefinidos;
2. Avaliar limiares operacionais de severidade:
   - Se `temperatura > 80` OU `vibracao > 7.0` -> Estado `CRÍTICO` (codigo de cor `#dc3545`);
   - Se `temperatura > 65` OU `vibracao > 4.5` -> Estado `ALERTA` (codigo de cor `#ffc107`);
   - Caso contrario -> Estado `NORMAL` (codigo de cor `#28a745`);
3. Enriquecer o objeto `msg` com novos metadados (`msg.status_geral`, `msg.cor_status` e a string estruturada `msg.payload_formatado`).

### 8.2 Codigo JavaScript do No: Tratamento de Telemetria

```javascript
const payload = msg.payload || {};
const temp = payload.temperatura || 0;
const vib = payload.vibracao || 0;
const pressao = payload.pressao || 0;

// Avaliacao de Criticidade
let statusGeral = "NORMAL";
let cor = "#28a745"; // Verde

if (temp > 80 || vib > 7.0) {
    statusGeral = "CRÍTICO";
    cor = "#dc3545"; // Vermelho
} else if (temp > 65 || vib > 4.5) {
    statusGeral = "ALERTA";
    cor = "#ffc107"; // Amarelo
}

msg.status_geral = statusGeral;
msg.cor_status = cor;
msg.payload_formatado = `[${payload.maquina || 'MAQ_01'}] Temp: ${temp.toFixed(1)}°C | Vib: ${vib.toFixed(2)} mm/s | Pressão: ${pressao.toFixed(1)} bar -> Status: ${statusGeral}`;

return msg;
```

In [ ]:
# Demonstracao de formatacao e avaliacao logica equivalente em Python
payload_exemplo = {"maquina": "PRENSA_CNC_01", "temperatura": 88.5, "vibracao": 8.2, "pressao": 6.0}

temp = payload_exemplo.get("temperatura", 0)
vib = payload_exemplo.get("vibracao", 0)
pressao = payload_exemplo.get("pressao", 0)

if temp > 80 or vib > 7.0:
    status_geral = "CRÍTICO"
    cor = "#dc3545"
elif temp > 65 or vib > 4.5:
    status_geral = "ALERTA"
    cor = "#ffc107"
else:
    status_geral = "NORMAL"
    cor = "#28a745"

payload_formatado = f"[{payload_exemplo.get('maquina', 'MAQ_01')}] Temp: {temp:.1f}°C | Vib: {vib:.2f} mm/s | Pressão: {pressao:.1f} bar -> Status: {status_geral}"
print("Saida gerada pela logica:")
print(payload_formatado)
print(f"Cor atribuida: {cor}")


### 8.3 Insercao e Conexao
1. Arraste um no **function** do grupo **Function** para o canvas.
2. Posicione-o a direita do no `Roteador por Topico/Tipo`, alinhado com a saida 1.
3. De um duplo clique sobre ele:
   - **Name:** `Tratamento de Telemetria`
   - **Outputs:** `1`
   - Cole o codigo JavaScript fornecido no item 8.2.
4. Clique em **Done**.
5. Conecte a **primeira porta de saida** do no `Roteador por Topico/Tipo` a entrada deste no de funcao.

--- 

## 9. Passo 7: No de Funcao 2 - Formatador de Alarmes Criticos

Este no processara mensagens originadas da rota de alarmes industriais (Porta 2 do no `switch`). A finalidade e padronizar a estrutura de dados para consumo por sistemas de manutencao ou notificacao.

### 9.1 Objetivo do Algoritmo
1. Atribuir o metadado prioritario `msg.prioridade = "MÁXIMA"`;
2. Criar a estrutura canonica `msg.payload_alarme` contendo tipo, origem, mensagem de texto, registro temporal ISO 8601 e recomendacao de acao operacional.

### 9.2 Codigo JavaScript do No: Formatador de Alarmes Criticos

```javascript
const payload = msg.payload || {};

msg.prioridade = "MÁXIMA";
msg.payload_alarme = {
    tipo: payload.tipo || "ALARME_GERAL",
    origem: payload.origem || msg.topic,
    mensagem: payload.mensagem || "Evento de emergência detectado!",
    timestamp: payload.timestamp || new Date().toISOString(),
    acao_recomendada: payload.acao || "Verificar imediatamente o painel da máquina"
};

return msg;
```

In [ ]:
# Demonstracao da estrutura canonica de alarmes em Python
import datetime

payload_alarme_raw = {"tipo": "E-STOP", "mensagem": "Pressao anomala no circuito primario"}
topico_origem = "fabrica/linha1/prensa01/alarmes"

payload_alarme = {
    "tipo": payload_alarme_raw.get("tipo", "ALARME_GERAL"),
    "origem": payload_alarme_raw.get("origem", topico_origem),
    "mensagem": payload_alarme_raw.get("mensagem", "Evento de emergencia detectado!"),
    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "acao_recomendada": payload_alarme_raw.get("acao", "Verificar imediatamente o painel da maquina")
}

print("Estrutura padronizada do alarme:")
print(json.dumps(payload_alarme, indent=2))


### 9.3 Insercao e Conexao
1. Arraste outro no **function** para o canvas, posicionando-o abaixo do no anterior.
2. De um duplo clique:
   - **Name:** `Formatador de Alarmes Críticos`
   - **Outputs:** `1`
   - Cole o codigo JavaScript fornecido no item 9.2.
3. Clique em **Done**.
4. Conecte a **segunda porta de saida** do no `Roteador por Topico/Tipo` a entrada deste no de funcao.

--- 

## 10. Passo 8: No de Funcao 3 - Processador de Producao e Qualidade

A Porta 3 do no `switch` recebe eventos relacionados ao avanco de contadores de producao. O no de funcao associado executa o calculo do indice de refugo e a taxa percentual de qualidade.

### 10.1 Objetivo do Algoritmo
1. Ler `total_produzido`, `pecas_boas` e inferir o `refugo` caso nao venha explicito (`total - boas`);
2. Tratar a divisao por zero para lotes em inicializacao (`total == 0`);
3. Montar o objeto `msg.resumo_producao` formatado com precisao decimal.

### 10.2 Codigo JavaScript do No: Processador de Producao

```javascript
const payload = msg.payload || {};
const total = payload.total_produzido || 0;
const boas = payload.pecas_boas || total;
const refugo = payload.refugo || (total - boas);
const taxaAprovacao = total > 0 ? ((boas / total) * 100).toFixed(1) : 100.0;

msg.resumo_producao = {
    maquina: payload.maquina || "LINHA_01",
    total_produzido: total,
    pecas_boas: boas,
    refugo: refugo,
    taxa_qualidade: `${taxaAprovacao}%`
};

return msg;
```

In [ ]:
# Demonstracao do calculo de qualidade de manufatura em Python
payload_producao_raw = {"maquina": "PRENSA_01", "total_produzido": 850, "pecas_boas": 825}

total = payload_producao_raw.get("total_produzido", 0)
boas = payload_producao_raw.get("pecas_boas", total)
refugo = payload_producao_raw.get("refugo", total - boas)
taxa_aprovacao = round((boas / total) * 100, 1) if total > 0 else 100.0

resumo_producao = {
    "maquina": payload_producao_raw.get("maquina", "LINHA_01"),
    "total_produzido": total,
    "pecas_boas": boas,
    "refugo": refugo,
    "taxa_qualidade": f"{taxa_aprovacao}%"
}

print("Resumo de producao consolidado:")
print(json.dumps(resumo_producao, indent=2))


### 10.3 Insercao e Conexao
1. Arraste mais um no **function** para o canvas, posicionando-o abaixo do no de alarmes.
2. De um duplo clique:
   - **Name:** `Processador de Produção`
   - **Outputs:** `1`
   - Cole o codigo JavaScript fornecido no item 10.2.
3. Clique em **Done**.
4. Conecte a **terceira porta de saida** do no `Roteador por Topico/Tipo` a entrada deste no de funcao.

--- 

## 11. Passo 9: Nos de Observabilidade Especializados (Debugs de Saida)

Para inspecionar as mensagens processadas por cada um dos canais sem misturar as propriedades no console lateral, instanciamos quatro nos de saida **debug**, parametrizando o caminho exato de inspecao.

### 11.1 Parametrizacao dos 4 Nos de Depuracao

1. **No 1 - Telemetria:**
   - **Name:** `[DEBUG] Telemetria Formatada`
   - **Output:** `msg.payload_formatado`
   - Conectado a saida do no `Tratamento de Telemetria`.

2. **No 2 - Alarmes:**
   - **Name:** `[DEBUG] Alarme Crítico / Emergência`
   - **Output:** `msg.payload_alarme`
   - Conectado a saida do no `Formatador de Alarmes Críticos`.

3. **No 3 - Producao:**
   - **Name:** `[DEBUG] Produção & Qualidade`
   - **Output:** `msg.resumo_producao`
   - Conectado a saida do no `Processador de Produção`.

4. **No 4 - Outros Topicos (Residual):**
   - **Name:** `[DEBUG] Outros Tópicos`
   - **Output:** `complete msg object` (selecionar opcao de mensagem completa)
   - Conectado diretamente a **quarta porta de saida** do no `Roteador por Topico/Tipo`.

--- 

## 12. Passo 10: Injetor de Carga para Testes Unitarios de Bancada (Inject)

Durante o desenvolvimento e homologacao de fluxos, testar a logica sem depender do publicador fisico e fundamental. O no **inject** permite disparar mensagens simuladas diretamente para a entrada do multiplexador.

### 12.1 Configuracao do No Inject
1. No grupo **Common**, arraste um no **inject** para o canvas.
2. Posicione-o na parte superior, acima do no `JSON Parser`.
3. De um duplo clique para abrir a edicao:
   - **Name:** `Injetar Teste Manual`
   - No campo **msg.payload**, altere o tipo para **JSON** (selecione `{}` no seletor suspenso) e insira o JSON abaixo:
     ```json
     {"maquina":"PRENSA_CNC_01","temperatura":88.5,"vibracao":8.2,"pressao":6.0,"status":"CRITICAL"}
     ```
   - Clique em **+ add** para adicionar uma nova propriedade na mensagem:
     - Propriedade: `msg.topic`
     - Tipo: `string` (az)
     - Valor: `fabrica/linha1/prensa01/telemetria`
   - **Repeat:** `none`
4. Clique em **Done**.

### 12.2 Conexao (Wire)
- Conecte a saida do no `Injetar Teste Manual` diretamente a entrada do no `Roteador por Topico/Tipo`.

--- 

## 13. Passo 11: Compilacao em Tempo de Execucao (Deploy) e Validacao

Toda alteracao feita na interface visual permanece no buffer local do navegador ate que a operacao de compilacao e ativacao (*Deploy*) seja acionada.

### 13.1 Modos de Deploy
Ao clicar na seta adjacente ao botao vermelho **Deploy** (canto superior direito), e possivel selecionar:
1. **Full (Completo):** Recompila e reinicia todos os nos da aplicacao.
2. **Modified Flows (Fluxos Modificados):** Reinicia apenas as abas que sofreram alteracoes.
3. **Modified Nodes (Apenas Nos Modificados):** Opcao mais rapida, reinicia estritamente os nos cujas configuracoes foram alteradas.

Clique no botao **Deploy**.

### 13.2 Inspecao no Painel Debug
1. Abra a aba lateral de depuracao no canto direito superior (atalho `Ctrl + G` seguido de `D`).
2. Caso o publicador MQTT da fabrica virtual ja esteja operando em segundo plano, mensagens comecarao a ser exibidas de forma continua.
3. Para validar o comportamento de falha isoladamente, clique no botao retangular a esquerda do no `Injetar Teste Manual`.
4. No painel de debug, confirme a recepcao da seguinte linha formatada:
   - `[PRENSA_CNC_01] Temp: 88.5°C | Vib: 8.20 mm/s | Pressão: 6.0 bar -> Status: CRÍTICO`

---

## 14. Teste Programatico de Disparo de Eventos via Python

A celula de codigo abaixo permite enviar pacotes MQTT controlados para o broker local utilizando a biblioteca `paho-mqtt`. Isso possibilita validar cada uma das tres rotas do `switch` sem depender exclusivamente de eventos externos da planta.

In [ ]:
# Disparo Programatico de Pacotes MQTT para Validacao do Pipeline Node-RED
import json
import time

try:
    import paho.mqtt.client as mqtt
    PAHO_DISPONIVEL = True
except ImportError:
    PAHO_DISPONIVEL = False

if PAHO_DISPONIVEL:
    BROKER_HOST = "localhost"
    BROKER_PORT = 1883
    
    cliente = mqtt.Client(client_id="injetor_validacao_semana07")
    try:
        cliente.connect(BROKER_HOST, BROKER_PORT, keepalive=60)
        print(f"Conectado com sucesso ao broker {BROKER_HOST}:{BROKER_PORT}")
        
        # 1. Envio de Telemetria com Sobreaquecimento
        payload_telemetria = {
            "maquina": "TORNO_CNC_02",
            "temperatura": 72.4,
            "vibracao": 4.9,
            "pressao": 5.2
        }
        topico_telemetria = "fabrica/linha1/torno02/telemetria"
        cliente.publish(topico_telemetria, json.dumps(payload_telemetria))
        print(f"[PUBLICADO] Topico: {topico_telemetria} | Payload: {payload_telemetria}")
        
        time.sleep(1.0)
        
        # 2. Envio de Alarme Critico de Parada
        payload_alarme = {
            "tipo": "E-STOP_ACIONADO",
            "origem": "BOTAO_EMERGENCIA_PAINEL",
            "mensagem": "Acionamento manual de parada na celula robotica",
            "acao": "Inspecionar celula e destravar rele de seguranca"
        }
        topico_alarme = "fabrica/linha1/celula_robo/alarmes"
        cliente.publish(topico_alarme, json.dumps(payload_alarme))
        print(f"[PUBLICADO] Topico: {topico_alarme} | Payload: {payload_alarme}")
        
        time.sleep(1.0)
        
        # 3. Envio de Dados de Producao e Refugo
        payload_producao = {
            "maquina": "PRENSA_ESTAMPAGEM_01",
            "total_produzido": 1250,
            "pecas_boas": 1218,
            "refugo": 32
        }
        topico_producao = "fabrica/linha2/prensa01/producao"
        cliente.publish(topico_producao, json.dumps(payload_producao))
        print(f"[PUBLICADO] Topico: {topico_producao} | Payload: {payload_producao}")
        
        cliente.disconnect()
        print("\nPacotes enviados com sucesso. Inspecione os 3 nós de Debug no Node-RED.")
    except Exception as e:
        print(f"Falha ao publicar no broker MQTT: {e}")
else:
    print("Biblioteca paho-mqtt nao encontrada no kernel Python.")
    print("O teste de bancada pode ser realizado diretamente pelo no 'Injetar Teste Manual' no Node-RED.")


--- 

## 15. Exercicios de Fixacao e Avaliacao Pratica

### Exercicio 1 (Limiares Dinamicos no No de Funcao)
Modifique o codigo do no `Tratamento de Telemetria` para incluir uma verificacao da grandeza fisica **pressao** (`pressao`):
- Se `pressao > 7.5` bar, force o estado para `CRÍTICO` independentemente da temperatura e defina a cor `#dc3545`.
- Se `pressao > 6.5` bar, defina o estado como `ALERTA` (caso ja nao esteja em nivel critico).
- Apresente o bloco de codigo JavaScript modificado e a saida gerada no no de depuracao.

### Exercicio 2 (Expansao de Rotas no No Switch)
Adicione uma nova rota na tabela do no `Roteador por Topico/Tipo` para mensagens de manutencao preventiva:
- Regra: `contains` -> `manutencao`.
- Conecte uma nova funcao que adicione `msg.ordem_servico = true` e exiba o resultado em um novo no de debug dedicado.

### Exercicio 3 (Analise Conceitual de Concorrencia)
Explique por que, no paradigma de programacao orientada a fluxos (FBP) sob o motor Node.js (single-threaded nao bloqueante com event-loop), multiplos eventos chegando simultaneamente pelo no `MQTT In` sao processados sem risco de colisao de variaveis locais declaradas com `const` ou `let` dentro dos nos `function`.